# March Machine Learning Mania 2026

|No| Name  | Notebook |
| --- | ------ | ------- |
| 00 | Dataset |  https://www.kaggle.com/competitions/march-machine-learning-mania-2026/data |
| 01 | Understand Data Structure  | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p1-dataset-overview-structure-understanding/ |
| 02 | Create a baseline model only on Male rqd. 4 datasets | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p2-basline-male-4-datasets/ |
| 03 | Features Eng | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p3-features-eng/ |
| 04 | Simple ELO | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p4-simple-elo/ |
| 05 | Adv ELO | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p5-adv-elo/  |
| | |  |

In [186]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

import optuna
from sklearn import set_config
set_config(display="diagram")

**Story**

Features

- Part 2 V1 - Baseline model (`win diff`, `seed diff`)
- Part 3 
  - V1 - `margin diff`
  - V2 - `recent win pct diff` Hypothesis: Teams playing well recently perform better than tournament
- Part 4
  - V1: Simple ELO + All Features
  - V2: Remove these 'Margin_Diff','RecentWinPct_Diff'still good result.
- Part 5
    -  V1: Add Home Advantage as a multiplier in the elo function

# Basic 

> - Again, without understanding the basic we can't code.
> - First, I am new to sports analysis.
> - So we need info and data to understand the game rules.
> - So if you new let's take the help of an infinitely intelligent


## What is ELO?

ELO is a dynamic rating system.

* Every team starts with same rating (ex: 1500)
* If strong team beats weak team → small rating change
* If weak team upsets strong team → big rating change

It automatically captures:

* strength
* upset power
* consistency
* schedule difficulty



## ELO Formula
- wiki: https://en.wikipedia.org/wiki/Elo_rating_system

For a match:

**Expected score:**

$$E_A = 1 / (1 + 10^{(R_B - R_A)/400}) $$

**Update:**

$$R_A = R_A + K * (S_A - E_A)$$
Where:

* `R_A` = rating of Team A
* `R_B` = rating of Team B
* `S_A` = 1 if win, 0 if loss
* `K` = learning rate (usually 20)

---


# Import Data

In [158]:
# Root file
data_file_path = r"/kaggle/input/march-machine-learning-mania-2026"
# Root file
data_file_path = r"C:\Users\Rudra\Desktop\kaggle\NCAA\data"

# Load only required files
regular = pd.read_csv(os.path.join(data_file_path, "MRegularSeasonCompactResults.csv"))
tourney = pd.read_csv(os.path.join(data_file_path,"MNCAATourneyCompactResults.csv"))
seeds = pd.read_csv(os.path.join(data_file_path,"MNCAATourneySeeds.csv"))
submission = pd.read_csv(os.path.join(data_file_path,"SampleSubmissionStage1.csv"))

# Sample View
display(regular.sample(3))
display(tourney.sample(3))
display(seeds.sample(3))
display(submission.sample(3))

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
124871,2013,25,1393,91,1116,82,A,0
179939,2023,94,1377,71,1295,62,H,0
92259,2006,121,1138,83,1103,71,H,0


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
2494,2024,138,1397,62,1400,58,N,0
1530,2009,136,1345,61,1320,56,N,0
2418,2023,137,1395,72,1113,70,N,0


,Season,Seed,TeamID
1628,2010,X04,1345
1998,2015,Z02,1112
640,1995,W01,1448


,ID,Pred
393723,2025_1119_1429,0.5
121158,2022_3358_3420,0.5
146592,2023_1165_1325,0.5


# clean Datasets

In [159]:
wins = regular.groupby(['Season', 'WTeamID']).size().reset_index(name='wins')
losses = regular.groupby(['Season', 'LTeamID']).size().reset_index(name='losses')

wins = wins.rename(columns={"WTeamID": 'TeamID'})
losses = losses.rename(columns={'LTeamID': 'TeamID'})

team_stats = pd.merge(
    wins, losses,
    how="outer", on=["Season", 'TeamID']
).fillna(0)

team_stats["TotalGames"] = team_stats["wins"] + team_stats["losses"]
team_stats['WinPct'] = team_stats['wins'] / team_stats['TotalGames']

display(team_stats.sample(3))


# Clean Seeds
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)
display(seeds.sample(3))


# Clean Tournament
tourney['Team1'] = tourney[['WTeamID', 'LTeamID']].min(axis=1)
tourney['Team2'] = tourney[['WTeamID', 'LTeamID']].max(axis=1)
tourney['Team1Win'] = (tourney['WTeamID'] == tourney['Team1']).astype(int)

train = tourney[['Season', 'Team1', 'Team2', 'Team1Win']]
display(train.sample(3))
print(train.shape)

,Season,TeamID,wins,losses,TotalGames,WinPct
11036,2019,1256,18.0,13.0,31.0,0.580645
8333,2011,1354,6.0,22.0,28.0,0.214286
6633,2006,1369,5.0,20.0,25.0,0.200000


,Season,Seed,TeamID,SeedNum
2145,2017,Z12,1292,12
2257,2019,Y04,1242,4
2486,2023,Z13,1233,13


,Season,Team1,Team2,Team1Win
1468,2008,1245,1424,0
1175,2003,1143,1328,0
797,1997,1155,1409,1


(2585, 4)


# Add Features

## ELO 


### Margin-Based ELO 

Right now:

Win by 1 point = Win by 30 points
That’s not realistic.

We modify K dynamically based on score difference.



### Margin Multiplier Formula

Common version:

$$ Multiplier = \log(|Margin| + 1) * \frac{2.2}{((R_w - R_l)*0.001 + 2.2)} $$

This means:

* Big win → bigger rating update
* Upset → bigger update
* Close win → smaller update


### ELO Calculator

In [160]:
def calculate_elo(df, k=20, initial_rating=1500):
    df = df.sort_values(['Season', 'DayNum'])
    
    final_elo = []

    for season in df['Season'].unique():
        
        season_games = df[df['Season'] == season]
        teams = pd.unique(season_games[['WTeamID','LTeamID']].values.ravel())
        
        # initialize ratings
        ratings = {team: initial_rating for team in teams}
        
        for _, row in season_games.iterrows():
            teamA = row['WTeamID']
            teamB = row['LTeamID']
            
            RA = ratings[teamA]
            RB = ratings[teamB]
            
            # home advantage
            home_adv = 100 if row['WLoc'] =='H' else 0
            if row['WLoc'] == 'A':
                home_adv =-100
                
            RA_adj = RA + home_adv
            
            # expected scores
            EA = 1 / (1 + 10 ** ((RB - RA_adj) / 400))
            EB = 1 - EA
            
            margin = row['WScore'] - row['LScore']
            
            multiplier = np.log(abs(margin) + 1) * (2.2 / ((RA - RB)*0.001 + 2.2))
                 
            # update ratings
            ratings[teamA] = RA + k * multiplier * (1 - EA)
            ratings[teamB] = RB + k * multiplier * (0 - EB)
        
        # save final ratings
        for team, rating in ratings.items():
            final_elo.append([season, team, rating])
    
    elo_df = pd.DataFrame(final_elo, columns=['Season','TeamID','ELO'])
    return elo_df

In [161]:
elo_ratings = calculate_elo(regular)

In [162]:
display(elo_ratings.sample(3))

,Season,TeamID,ELO
3870,1998,1254,1635.493103
5311,2002,1460,1517.305228
12457,2023,1358,1693.379074


In [163]:
def add_feature_elo(df):
    # team1 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team1_ELO'}).drop('TeamID', axis=1)

    # team2 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team2_ELO'}).drop('TeamID', axis=1)
    
    return df

## Add Score Margin

In [164]:
regular['ScoreDiff'] = regular['WScore'] - regular['LScore']

# For Winners
w_margin = regular.groupby(['Season', 'WTeamID'])['ScoreDiff'].mean().reset_index()
w_margin = w_margin.rename(columns={'WTeamID':'TeamID', 'ScoreDiff':'AvgMargin'})

# For Losers
l_margin = regular.groupby(['Season','LTeamID'])['ScoreDiff'].mean().reset_index()
l_margin['ScoreDiff'] = -l_margin['ScoreDiff']
l_margin = l_margin.rename(columns={'LTeamID':'TeamID','ScoreDiff':'AvgMargin'})

# Combine
margin = pd.concat([w_margin, l_margin])
margin = margin.groupby(['Season','TeamID'])['AvgMargin'].mean().reset_index()
display(margin.sample(3))

,Season,TeamID,AvgMargin
842,1987,1449,2.070513
4849,2001,1193,-1.375000
11461,2020,1331,4.228022


## Add Recent Win Pct

In [165]:
regular_recent =  regular[regular['DayNum'] > 100].copy()

# wins
wins_recent = regular_recent.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
wins_recent = wins_recent.rename(columns={'WTeamID': 'TeamID'})

# Losses
losses_recent = regular_recent.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
losses_recent = losses_recent.rename(columns={'LTeamID': 'TeamID'})

recent_stats = pd.merge(
    wins_recent, losses_recent,
    on=['Season', 'TeamID'],
    how='outer'
).fillna(0)

recent_stats['Total'] = recent_stats['Wins'] + recent_stats['Losses']
recent_stats['RecentWinPct'] =  recent_stats['Wins'] / recent_stats['Total']

recent_stats.head(3)


,Season,TeamID,Wins,Losses,Total,RecentWinPct
0,1985,1102,3.0,5.0,8.0,0.375000
1,1985,1103,3.0,4.0,7.0,0.428571
2,1985,1104,7.0,2.0,9.0,0.777778


In [166]:
def add_feature_recent_win_pct(df):
    # recent team 1 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team1_RecentWinPct'}).drop('TeamID', axis=1)

    # recent team 2 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team2_RecentWinPct'}).drop('TeamID', axis=1)


    return df

## Add Win Pct Diff 

In [167]:
def add_feature_win_pct(df:pd.date_range) -> pd.DataFrame:
    """Create features for both train and test(submission) data."""

    # Team1 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team1_WinPct'}).drop('TeamID', axis=1)

    # Team2 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team2_WinPct'}).drop('TeamID', axis=1)
    
    return df

## Add Seed

In [168]:

def add_feature_seed(df):
    # team1 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team1_Seed'}).drop('TeamID', axis=1)

    # team 2 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team2_Seed'}).drop('TeamID', axis=1)
    
    return df

## Add Avg Margin

In [169]:
def add_feature_avg_margin(df):
    # team 1 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team1_AvgMargin'}).drop('TeamID', axis=1)
    
    # team 2 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team2_AvgMargin'}).drop('TeamID', axis=1)

    return df

In [170]:
def add_all_features(df):
    """Call al the features function at a time."""
    df = add_feature_win_pct(df)
    df = add_feature_seed(df)
    # df = add_feature_avg_margin(df)
    # df = add_feature_recent_win_pct(df)
    df = add_feature_elo(df)
    
    return df

In [171]:
train = add_all_features(train)

# Create Model Inputs Features

In [172]:
def create_diff(df:pd.DataFrame) ->pd.DataFrame:
    """ Create Features difference."""
    
    df['WinPct_Diff'] = df['Team1_WinPct'] - df['Team2_WinPct']

    df['Seed_Diff'] = df['Team2_Seed'] - df['Team1_Seed'] 

    # df['Margin_Diff'] = df['Team1_AvgMargin'] - df['Team2_AvgMargin']
    # df['RecentWinPct_Diff'] = df['Team1_RecentWinPct'] - df['Team2_RecentWinPct']

    df['ELO_Diff'] =  df['Team1_ELO'] - df['Team2_ELO']
    df['ELO_Prob'] = 1 / (1 + 10 ** (-df['ELO_Diff'] / 400))
    
    return df

In [173]:
train = create_diff(train)

train.sample(3)

,Season,Team1,Team2,Team1Win,Team1_WinPct,Team2_WinPct,Team1_Seed,Team2_Seed,Team1_ELO,Team2_ELO,WinPct_Diff,Seed_Diff,ELO_Diff,ELO_Prob
2393,2023,1222,1297,1,0.911765,0.625000,1,16,1836.815226,1629.407059,0.286765,15,207.408168,0.767445
122,1986,1242,1301,1,0.909091,0.571429,1,6,1865.673212,1628.314274,0.337662,5,237.358938,0.796789
1515,2008,1172,1242,0,0.806452,0.909091,10,1,1812.583147,1882.209525,-0.102639,-9,-69.626378,0.401120


# Model

In [174]:
# model_train_input_cols = ['WinPct_Diff','Seed_Diff','Margin_Diff','RecentWinPct_Diff', 'ELO_Diff']
model_train_input_cols = ['WinPct_Diff','Seed_Diff', 'ELO_Diff']

In [175]:
train_data = train[train['Season'] < 2024]
val_data   = train[train['Season'] == 2024]

X_train = train_data[model_train_input_cols]
y_train = train_data['Team1Win']

X_val = val_data[model_train_input_cols]
y_val = val_data['Team1Win']

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

model_prob_val = model.predict_proba(X_val)[:,1]
elo_prob_val = 1 / (1 + 10 ** (-X_val['ELO_Diff'] / 400))


best_loss = 999
best_weight = None

for w in np.arange(0, 1.01, 0.05):
    
    blend = w * elo_prob_val + (1 - w) * model_prob_val
    loss = log_loss(y_val, blend)
    
    print(f"Weight: {w:.2f} -> LogLoss: {loss:.5f}")
    
    if loss < best_loss:
        best_loss = loss
        best_weight = w

print(f"\n\n\nBest Weight: {best_weight}")
print(f"Best Log Loss: {best_loss}")


Weight: 0.00 -> LogLoss: 0.56177
Weight: 0.05 -> LogLoss: 0.56217
Weight: 0.10 -> LogLoss: 0.56289
Weight: 0.15 -> LogLoss: 0.56391
Weight: 0.20 -> LogLoss: 0.56522
Weight: 0.25 -> LogLoss: 0.56681
Weight: 0.30 -> LogLoss: 0.56867
Weight: 0.35 -> LogLoss: 0.57078
Weight: 0.40 -> LogLoss: 0.57314
Weight: 0.45 -> LogLoss: 0.57576
Weight: 0.50 -> LogLoss: 0.57861
Weight: 0.55 -> LogLoss: 0.58171
Weight: 0.60 -> LogLoss: 0.58506
Weight: 0.65 -> LogLoss: 0.58864
Weight: 0.70 -> LogLoss: 0.59247
Weight: 0.75 -> LogLoss: 0.59654
Weight: 0.80 -> LogLoss: 0.60087
Weight: 0.85 -> LogLoss: 0.60545
Weight: 0.90 -> LogLoss: 0.61029
Weight: 0.95 -> LogLoss: 0.61540
Weight: 1.00 -> LogLoss: 0.62078



Best Weight: 0.0
Best Log Loss: 0.5617698575471601


# Submission

In [177]:
def create_id(df):
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True)

    df['Season'] = df['Season'].astype(int)
    df['Team1'] = df['Team1'].astype(int)
    df['Team2'] = df['Team2'].astype(int)

    return df

submission = create_id(submission)
submission = add_all_features(submission)
submission = create_diff(submission)

# Fill missing features
submission[model_train_input_cols] = submission[model_train_input_cols].fillna(0)


In [178]:
submission['ELO_Prob'] = 1 / (1 + 10 ** (-submission['ELO_Diff'] / 400))

model_prob = model.predict_proba(
    submission[model_train_input_cols]
)[:,1]

submission['Pred'] = best_weight * submission['ELO_Prob'] + \
                     (1 - best_weight) * model_prob

In [183]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

def objective(trial):
    
    C = trial.suggest_float("C", 0.001, 10, log=True)
    penalty = trial.suggest_categorical("penalty", ["l2"])
    max_iter = trial.suggest_int("max_iter", 1000, 10_000)
    
    model = LogisticRegression(
        C=C,
        penalty=penalty,
        max_iter=max_iter,
        solver="lbfgs"
    )
    
    model.fit(X_train, y_train)
    
    model_prob_val = model.predict_proba(X_val)[:,1]
    loss = log_loss(y_val, model_prob_val)
    
    return loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best loss:", study.best_value)

[I 2026-02-28 07:47:48,764] A new study created in memory with name: no-name-d4ab3aa0-8b1c-4f2f-b4fb-bfef803af22a
[I 2026-02-28 07:47:48,807] Trial 0 finished with value: 0.5635971309112534 and parameters: {'C': 0.0070382416919541535, 'penalty': 'l2', 'max_iter': 9772}. Best is trial 0 with value: 0.5635971309112534.
[I 2026-02-28 07:47:48,852] Trial 1 finished with value: 0.5636445812890709 and parameters: {'C': 0.0047572783781265, 'penalty': 'l2', 'max_iter': 7873}. Best is trial 0 with value: 0.5635971309112534.
[I 2026-02-28 07:47:48,895] Trial 2 finished with value: 0.5615603775261121 and parameters: {'C': 3.450135607963168, 'penalty': 'l2', 'max_iter': 4206}. Best is trial 2 with value: 0.5615603775261121.
[I 2026-02-28 07:47:48,936] Trial 3 finished with value: 0.5615840196003024 and parameters: {'C': 2.5896965847262465, 'penalty': 'l2', 'max_iter': 5418}. Best is trial 2 with value: 0.5615603775261121.
[I 2026-02-28 07:47:48,974] Trial 4 finished with value: 0.5635489039473912 

Best params: {'C': 9.927808074692017, 'penalty': 'l2', 'max_iter': 4867}
Best loss: 0.5615256321983321


In [ ]:
best_params = study.best_params

model = LogisticRegression(
    C=best_params["C"],
    penalty=best_params["penalty"],
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)


# Logistic prediction
model_prob = model.predict_proba(
    submission[model_train_input_cols]
)[:, 1]

# Blend
submission['Pred'] = 0.5 * submission['ELO_Prob'] + 0.5 * model_prob

# Export
submission[['ID','Pred']].to_csv("submission.csv", index=False)

print("Rows in submission:", len(submission))
print("Submission exported successfully.")
display(submission[['ID','Pred']].head(3))

LogisticRegression(C=9.927808074692017, max_iter=1000)

In [185]:
# Logistic prediction
model_prob = model.predict_proba(
    submission[model_train_input_cols]
)[:, 1]

# Blend
submission['Pred'] = 0.5 * submission['ELO_Prob'] + 0.5 * model_prob

# Export
submission[['ID','Pred']].to_csv("submission.csv", index=False)

print("Rows in submission:", len(submission))
print("Submission exported successfully.")
display(submission[['ID','Pred']].head(3))

Rows in submission: 519144
Submission exported successfully.


,ID,Pred
0,2022_1101_1102,0.764550
1,2022_1101_1103,0.401781
2,2022_1101_1104,0.543433
